In [1]:
import sys
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from datasets import Dataset
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
from transformers import TrainingArguments, Trainer
import numpy as np
from transformers import DataCollatorWithPadding


c:\VSCode Python\FakeNewsDetection\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# 1. Load and Prepare Data
big_path = "Data/big_data.csv"
smaller_path = "Data/smaller_data.csv"

df = pd.read_csv(smaller_path)
df.rename(columns={'label': 'labels'}, inplace=True)

# Convert to Hugging Face Dataset
hf_dataset = Dataset.from_pandas(df)

# Split into 80% training and 20% testing FIRST
split_datasets = hf_dataset.train_test_split(test_size=0.2, seed=42)

# 2. Tokenization Setup
# Removed the conflicting AutoTokenizer lines that referenced an undefined model_id
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize_function(examples):
    return tokenizer(
        examples["text"], 
        truncation=True, 
        max_length=512 # DistilBERT cannot handle 1024 tokens
    )

# Apply tokenization to the already-split datasets
tokenized_datasets = split_datasets.map(tokenize_function, batched=True)

Map: 100%|██████████| 3000/3000 [00:00<00:00, 10101.00 examples/s]


In [3]:
# Load DistilBERT model
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Metrics function
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    return {"accuracy": acc}

# Training settings (original style)
training_args = TrainingArguments(
    output_dir="./distilbert_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    load_best_model_at_end=True
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"], # Updated
    eval_dataset=tokenized_datasets["test"],   # Updated
    compute_metrics=compute_metrics,
    data_collator=data_collator
)

c:\VSCode Python\FakeNewsDetection\.venv\lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Jack\.cache\huggingface\hub\models--distilbert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 100/100 [00:00<00:00, 6667.04it/s]
DistilBertForSequenceClassification L

In [4]:
# Train model
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.123726,0.146823,0.966667
2,0.048348,0.134563,0.969333


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  5.26it/s]
There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=3000, training_loss=0.115177734375, metrics={'train_runtime': 1024.2659, 'train_samples_per_second': 23.431, 'train_steps_per_second': 2.929, 'total_flos': 2142800919871872.0, 'train_loss': 0.115177734375, 'epoch': 2.0})

In [ ]:
#LOADING MODEL, PRETRAINED
from transformers import AutoModelForSequenceClassification, AutoTokenizer

save_path = "./modernBERT-final"

my_model = AutoModelForSequenceClassification.from_pretrained(save_path)
my_tokenizer = AutoTokenizer.from_pretrained(save_path)

In [5]:
test_results = trainer.predict(tokenized_datasets["test"])

# The rest of the code remains exactly the same
predicted_labels = np.argmax(test_results.predictions, axis=-1)
actual_labels = test_results.label_ids

print(classification_report(actual_labels, predicted_labels, target_names=["real", "fake"]))

              precision    recall  f1-score   support

        real       0.99      0.95      0.97      1489
        fake       0.95      0.99      0.97      1511

    accuracy                           0.97      3000
   macro avg       0.97      0.97      0.97      3000
weighted avg       0.97      0.97      0.97      3000



In [ ]:
save_path = "./distilBERT-final"

trainer.save_model(save_path)

tokenizer.save_pretrained(save_path)